# 03 GMM with time-ordered grid search

**Objective:** Fit Gaussian mixture models to daily stock observations to identify groups with similar patterns in returns, momentum, volatility and trading activity. We compare models with different number of groups and use chronological validation to choose a suitable number. There are no predefined "correct" group labels to compare against. 

GMM describes the data as a combination of several Gaussian, or bell-shaped, distributions. Each distribution is called a component and represents a possible group of observations. Components can overlap, so the model assigns each observation a probability of belonging to each group.

This is a simplified representation of real financial data. Returns can have heavy tails and are not necessarily normally distributed. Combining several Gaussian components can approximate a more complex distribution, but the groups still need to be examined to understand what they represent.


`n_components` sets how many groups the GMM tries to identify. `covariance_type` determines how the model describes variation within each group: `diag` estimates a separate variance for each feature within each component; `full` also models correlations but requires more parameters. `reg_covar=1e-5` stabilizes the covariance matrices, `n_init` reduces sensitivity to initialization, and `max_iter` limits the number of fitting iterations.

We use a simple loop to try different GMM settings, such as the number of components and the covariance type. Each combination is evaluated using the same walk-forward splits: train on 2017 and validate on 2018, train on 2017-2018 and validate on 2019, then train on 2017-2019 and validate on 2020. Training always uses earlier observations, and all stocks observed on one date remain together. The imputer and scaler are fitted only on the training rows within each fold. 

Because we do not know the correct cluster labels, we cannot evaluate the model using classification accuracy. Instead, we use validation log-likelihood to measure how well the fitted GMM described later observations. Higher scores indicate a better accuracy fit, although resulting states (the groups of stock-day observations identified by the GMM) must also be checked for stability and interpretability. 

GridSearchCV could perform this search if we supplied suitable date-based splits and scoring settings. We use an explicit loop to make the training periods, preprocessing steps and evaluation process easier to inspect.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src/config.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src/config.py").exists():
    raise FileNotFoundError("Open this notebook from the project root or notebooks folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src.config import ProjectConfig, options
from src.io_utils import prepare_run, read_json, write_json, require_files, ensure_unlocked, save_figure
from src.data import load_development_data, make_feature_sets, make_preprocessor, chronological_folds

cfg = ProjectConfig()
OUT = prepare_run(ROOT, cfg)
ensure_unlocked(OUT)
